# Filters and Transitions in VideoDB Editor
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/editor/feature/filters_and_transitions.ipynb)

In this notebook, we'll explore how to apply **filters** and **transitions** to video clips using VideoDB Editor.

## What Are Filters?

Filters are color treatments that change the mood and visual appearance of your clips. Think of them as Instagram-style effects that you can apply programmatically—greyscale for a classic feel, blur for artistic effects, or negative for surreal visuals.

## What Are Transitions?

Transitions control how clips appear and disappear on the timeline. Instead of abrupt cuts, we can use smooth fade-in and fade-out effects that add professional polish to our videos.

## Why This Matters

Without filters and transitions, video edits can feel jarring and unpolished. These effects help us:
- Set the mood and tone of our content
- Create smooth scene changes
- Add cinematic quality without needing a GUI editor
- Build professional-looking videos entirely with code

## What We'll Cover

In this notebook, we will:
1. Apply all 8 available filters (greyscale, blur, contrast, darken, lighten, boost, muted, negative)
2. Demonstrate fade, reveal, and shuffle transitions
3. Control transition timing and duration
4. Combine filters with transitions for advanced effects
5. Show sequential clips with smooth transitions between scenes
6. Build a final showcase comparing all 8 transition types

Let's get started!

---

## 📦 Step 1: Installing VideoDB Editor SDK

First, we need to install the VideoDB Python SDK with Editor support. Run the cell below to install it quietly.

In [ ]:
!pip -q install videodb


---

## 📦 Step 2: Connecting to VideoDB

Now we'll establish a connection to VideoDB using your API key. The key will be requested securely and won't be visible in the notebook output.

In [ ]:
import videodb
import os
from getpass import getpass

api_key = getpass("Please enter your VideoDB API Key: ")

os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()

coll = conn.get_collection()

print("✅ Connected to VideoDB successfully!")

Please enter your VideoDB API Key: ··········
✅ Connected to VideoDB successfully!


---

## 📦 Step 3: Uploading Video Assets

For this notebook, we'll upload multiple videos to demonstrate different filters and transitions. We'll upload:
- A **main video** with good color variety (landscapes work great for showing filter effects)
- A **second video** for transition demonstrations between clips

Run the cell below to upload the first video from YouTube.

In [ ]:
# Upload first video (use a video with vibrant colors - landscapes, nature scenes work well)
video1 = coll.upload(url="https://www.youtube.com/watch?v=wU0PYcCsL6o")
print(f"✅ Uploaded video 1: {video1.id}")

# If you've already uploaded this video, use this instead:
# video1 = coll.get_video("your_video_id_here")

✅ Uploaded video 1: m-z-019edf2f-5214-7c93-990a-06224f107d75


Now let's upload a second video for demonstrating transitions between different clips.

In [ ]:
# Upload second video (any video with different content from the first)
video2 = coll.upload(url="https://www.youtube.com/watch?v=LejnTJL173Y")
print(f"✅ Uploaded video 2: {video2.id}")

# If you've already uploaded this video, use this instead:
# video2 = coll.get_video("your_video_id_here")

✅ Uploaded video 2: m-z-019edf30-5a7c-7ea3-9626-a6fa43a6b49a


---

## 📦 Step 4: Importing Editor Components

Let's import all the Editor components we'll need for this notebook: Timeline, Track, Clip, VideoAsset, Filter, Transition, and the play_stream function to preview our results.

In [ ]:
from videodb import play_stream
from videodb.editor import Timeline, Track, Clip, VideoAsset, TextAsset, Font, Background, TextAlignment, Filter, Transition, ImageAsset

print("✅ Editor components imported successfully!")

✅ Editor components imported successfully!


---

## 📦 Step 5: Understanding Filters

Filters are **clip-level effects** that change the visual appearance of your content. VideoDB Editor provides 8 different filters:

| Filter | Effect |
|--------|--------|
| `Filter.greyscale` | Removes all color, creating a black-and-white look |
| `Filter.blur` | Blurs the scene for artistic or privacy effects |
| `Filter.contrast` | Increases contrast, making darks darker and lights lighter |
| `Filter.darken` | Darkens the entire scene |
| `Filter.lighten` | Lightens the entire scene |
| `Filter.boost` | Boosts both contrast and saturation for vibrant colors |
| `Filter.muted` | Reduces saturation and contrast for a subdued look |
| `Filter.negative` | Inverts colors for a surreal, negative effect |

Filters are applied at the **Clip level**, not the Asset level. This means the same video asset can be used with different filters in different clips.

---

## 📦 Step 6: Applying a Basic Filter (Greyscale)

Let's start with the most common filter: greyscale. We'll apply it to our first video for 10 seconds.

Notice how we pass `filter=Filter.greyscale` to the Clip (not the VideoAsset). The timeline uses a neutral gray background (`#2B2B2B`) to focus on the content.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    filter=Filter.greyscale
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/a9644e38-0e49-4876-a8ee-1d8c69f0e633.m3u8


---

## 📦 Step 7: Showcasing All 8 Filters Sequentially

Now let's create a single video that shows all 8 filters one after another. Each filter will be displayed for 5 seconds, giving us enough time to see the effect clearly.

This is a great way to compare filters and understand their visual impact on the same source material.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

track = Track()

filters_list = [
    Filter.greyscale,
    Filter.blur,
    Filter.contrast,
    Filter.darken,
    Filter.lighten,
    Filter.boost,
    Filter.muted,
    Filter.negative
]

filter_names = [
    "Greyscale",
    "Blur",
    "Contrast",
    "Darken",
    "Lighten",
    "Boost",
    "Muted",
    "Negative"
]

start_time = 0
for filter_type, filter_name in zip(filters_list, filter_names):
    clip = Clip(
        asset=VideoAsset(id=video1.id, start=10),
        duration=5,
        filter=filter_type
    )
    track.add_clip(start_time, clip)
    start_time += 5

timeline.add_track(track)

text_track = Track()
start_time = 0
for filter_name in filter_names:
    text_asset = TextAsset(
        text=filter_name,
        font=Font(family="Clear Sans", size=36, color="#FFFFFF"),
        background=Background(
            color="#000000",
            opacity=0.6,
            height=60,
            width=200,
            text_alignment=TextAlignment.center
        )
    )
    text_clip = Clip(
        asset=text_asset,
        duration=5
    )
    text_track.add_clip(start_time, text_clip)
    start_time += 5

timeline.add_track(text_track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/863701c0-95c7-4b22-8e80-676a664af531.m3u8


<div style="background-color: #d4edda; color: #155724; padding: 12px; border-left: 5px solid #28a745; border-radius: 4px;">
    <strong>🎉 Milestone!</strong> We've demonstrated all 8 available filters in a single video! Each filter runs for 5 seconds using the same source segment, making it easy to compare their visual effects side by side.
</div>

> **What just happened?** We built a 40-second video (8 filters × 5 seconds each) by looping over a list of filter types and creating a clip for each one on the same track. Since they're on the same track, they play sequentially. This pattern is useful for filter previews, effect comparisons, or creating varied visual segments.


**Tip:** In the video above, you'll see each filter applied for 5 seconds. Notice how:
- **Greyscale** creates a timeless, classic look
- **Blur** softens the scene (great for backgrounds or privacy)
- **Contrast** makes the image more dramatic
- **Darken** and **Lighten** adjust brightness
- **Boost** makes colors pop
- **Muted** creates a subtle, understated feel
- **Negative** inverts everything for artistic effects

---

## 📦 Step 8: Combining Filters with Other Clip Properties

Filters work seamlessly with other clip properties like `scale` and `opacity`. Let's apply a greyscale filter combined with reduced opacity and slight scaling to create a faded, vintage look.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    filter=Filter.greyscale,
    scale=1.2,
    opacity=0.7
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/d8535e40-29ac-4a4a-a188-c753b05b376a.m3u8


**Note:** This combination of greyscale + opacity + scale creates a dreamy, vintage aesthetic. The gray background shows through the semi-transparent clip, giving it a faded look.

---

## 📦 Step 9: Understanding Transitions

Transitions control how clips **appear** and **disappear** on the timeline. Instead of abrupt cuts, we can use smooth animations.

VideoDB Editor supports **8 transition types** that control how clips enter and exit:

| Transition | Description |
|-----------|-------------|
| `fade` | Fade to/from black (classic smooth transition) |
| `reveal` | Slide reveal from edge |
| `wipe_left` | Wipe/sweep from the left edge |
| `wipe_right` | Wipe/sweep from the right edge |
| `shuffle_top_right` | Shuffle from top-right corner |
| `shuffle_bottom_right` | Shuffle from bottom-right corner |
| `shuffle_bottom_left` | Shuffle from bottom-left corner |
| `shuffle_top_left` | Shuffle from top-left corner |

Each transition uses these parameters:
- **in_**: Controls the entrance animation (note the underscore, since `in` is a Python keyword)
- **out**: Controls the exit animation
- **duration**: How long the transition lasts (in seconds, default: 0.5)

Transitions are applied at the **Clip level** using the `Transition` object.

---

## 📦 Step 10: Basic Fade In and Fade Out

Let's apply both fade-in and fade-out transitions to a clip. We'll use a 2-second duration for each transition, which provides a smooth, professional look.

Notice that we start the clip at 2 seconds on the timeline (`track.add_clip(2, clip)`) to see the fade-in effect clearly.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    transition=Transition(in_="fade", out="fade", duration=2)
)

track = Track()
track.add_clip(2, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/18c88b47-7df5-4400-b8c4-0d0c3962e38a.m3u8


**Tip:** The clip will:
1. Start at 2 seconds on the timeline
2. Fade in from transparent to full opacity over 2 seconds
3. Play normally for 6 seconds (10 - 2 - 2)
4. Fade out to transparent over the final 2 seconds

> **What just happened?** We applied both fade-in and fade-out transitions to a single clip. The `duration=2` means each transition takes 2 seconds — the first 2 seconds fade in, the last 2 seconds fade out, and the middle 6 seconds play at full opacity. This is the most common transition pattern for professional-looking video edits.


---

## 📦 Step 11: Transition Variations (Only In or Only Out)

We don't always need both transitions. Let's try using only a fade-in transition without a fade-out.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    transition=Transition(in_="fade", duration=2)
)

track = Track()
track.add_clip(2, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/f9b2fb12-53cc-4a18-b834-b68c60d16f50.m3u8


Now let's try the opposite—only a fade-out transition.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    transition=Transition(out="fade", duration=2)
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/31b6b0ff-d274-4a16-9ed2-6d9bbd8f5bfc.m3u8


> **What just happened?** Transitions can be applied independently — you can have just a fade-in (entrance), just a fade-out (exit), or both. Use `in_` and `out` separately when you only want animation on one side. This is useful for intro clips (fade-in only) or outro clips (fade-out only).


---

## 📦 Step 11.5: Reveal Transition

Beyond fade, VideoDB also supports **reveal** transitions — the clip slides in from the edge of the frame. Let's try it:

The `reveal` transition creates a smooth sliding entrance/exit effect, perfect for modern, dynamic content.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    transition=Transition(in_="reveal", out="reveal", duration=2)
)

track = Track()
track.add_clip(2, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/29552263-c6c2-411a-88db-209e438bc4b7.m3u8


> **What just happened?** The `reveal` transition slides the clip in from the edge when it enters and slides it out when it exits. Unlike `fade` which works on opacity, `reveal` is a motion-based transition that gives a polished, contemporary feel.


---

## 📦 Step 11.6: Shuffle Transitions

VideoDB also supports **shuffle** and **wipe** transitions that create dynamic tiling and sweeping effects. There are 6 variants:

| Transition | Origin |
|-----------|--------|
| `wipe_left` | Sweeps in from the left |
| `wipe_right` | Sweeps in from the right |
| `shuffle_top_right` | Top-right corner |
| `shuffle_bottom_right` | Bottom-right corner |
| `shuffle_bottom_left` | Bottom-left corner |
| `shuffle_top_left` | Top-left corner |

Let's showcase all 5 shuffle variants in sequence:

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

track = Track()

shuffle_types = [
    "wipe_left",
    "wipe_right",
    "shuffle_top_right",
    "shuffle_bottom_right",
    "shuffle_bottom_left",
    "shuffle_top_left",
]

start_time = 0
for st in shuffle_types:
    clip = Clip(
        asset=VideoAsset(id=video1.id, start=10),
        duration=5,
        transition=Transition(in_=st, out=st, duration=1)
    )
    track.add_clip(start_time, clip)
    start_time += 5

timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/d769c6ce-861a-4fc4-b19b-b9f384590870.m3u8


> **What just happened?** The `wipe_left` and `wipe_right` transitions sweep across the screen, while the `shuffle_*` variants tile in from specific corners. Each creates a different feel — wipes are great for scene changes in presentations or slideshows, while shuffles work well for energetic social media content.


---

## 📦 Step 12: Transitions Work with All Asset Types

Transitions aren't limited to video clips. Let's upload an image and apply a fade transition to it.

In [ ]:
# Upload an image
image = coll.upload(url="https://images.unsplash.com/photo-1506905925346-21bda4d32df4")
print(f"✅ Uploaded image: {image.id}")

# If you've already uploaded this image:
# image = coll.get_image("your_image_id_here")

✅ Uploaded image: img-z-019edf31-4b48-7621-8b17-a03208b0d1be


Now let's apply fade transitions to the image clip.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=ImageAsset(id=image.id),
    duration=8,
    transition=Transition(in_="fade", out="fade", duration=2)
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/f76ea973-7ee9-488c-a2bc-ea7c480d3a54.m3u8


> **What just happened?** Transitions work with all asset types — not just video. We applied the same fade transitions to an image clip and it worked seamlessly. This means you can use fade in/out for slideshows, image-based title cards, or any static visual content.


---

## 📦 Step 13: Sequential Clips with Smooth Transitions

One of the most powerful uses of transitions is creating smooth scene changes between different video clips. Let's create a sequence using our two uploaded videos with fade transitions between them.

This creates a professional-looking video montage without any jarring cuts.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

track = Track()

# First clip with fade-out
clip1 = Clip(
    asset=VideoAsset(id=video1.id),
    duration=8,
    transition=Transition(out="fade", duration=2)
)

# Second clip with fade-in and fade-out
clip2 = Clip(
    asset=VideoAsset(id=video2.id),
    duration=8,
    transition=Transition(in_="fade", out="fade", duration=2)
)

# Third clip with fade-in
clip3 = Clip(
    asset=VideoAsset(id=video1.id, start=20),
    duration=8,
    transition=Transition(in_="fade", duration=2)
)

track.add_clip(0, clip1)
track.add_clip(8, clip2)
track.add_clip(16, clip3)

timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/03ffbfaa-b7c0-40e1-a939-f2dd4bcdc25f.m3u8


**Note:** This creates a 24-second video with three scenes:
- **0-8s:** First video fades out at the end
- **8-16s:** Second video fades in at the start and fades out at the end
- **16-24s:** First video (different segment) fades in at the start

The transitions create smooth, professional scene changes.

<div style="background-color: #d4edda; color: #155724; padding: 12px; border-left: 5px solid #28a745; border-radius: 4px;">
    <strong>✅ Major Milestone!</strong> You've created a multi-clip video montage with smooth fade transitions between each scene. Each clip has its own transition profile (fade-out only, both, fade-in only), yet they all flow together seamlessly.
</div>

> **What just happened?** We built a 3-scene montage using two different videos. Scene 1 fades out at the end, Scene 2 fades in then out, and Scene 3 fades in at the start. The result is a polished, professional video with no jarring cuts — all done in a few lines of code.


---

## 📦 Step 14: Combining Filters and Transitions

Now for the best part—we can combine filters and transitions on the same clip. Let's create a dramatic effect with greyscale filter and fade transitions.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=12,
    filter=Filter.greyscale,
    transition=Transition(in_="fade", out="fade", duration=2)
)

track = Track()
track.add_clip(2, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/0ce7ce01-72e1-4371-b491-2f5936f05a5f.m3u8


**Tip:** This creates a cinematic black-and-white sequence that fades in and out smoothly. Perfect for dramatic flashbacks or artistic segments!

> **What just happened?** Filters and transitions are independent clip-level properties that can be combined freely. The greyscale filter changes the visual appearance, while the fade transitions control how the clip enters and exits. They don't interfere with each other — you get both effects working together.


---

## 📦 Step 15: Advanced Effect Stacking

Let's push the limits by combining multiple effects: filter, transition, scale, and opacity. This creates a sophisticated, layered visual effect.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    filter=Filter.muted,
    transition=Transition(in_="fade", out="fade", duration=2),
    scale=1.3,
    opacity=0.8
)

track = Track()
track.add_clip(2, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/05330871-9bb2-41b3-aabd-8cb236cb9dc9.m3u8


**What's happening here:**
- **Muted filter** reduces saturation and contrast for a subtle, dreamy look
- **Fade transitions** (2 seconds each) create smooth entry and exit
- **Scale 1.3** zooms in slightly, creating a Ken Burns effect
- **Opacity 0.8** makes the clip semi-transparent, revealing the gray background

This combination creates a sophisticated, artistic effect perfect for montages or memory sequences.

---

## 📦 Step 15.5: Transition Showcase — All 8 Types with a Filter

Now let's combine everything we've learned. We'll apply a single filter (`boost` for vibrant colors) to three different clips, each using a **different transition type**. This demonstrates the full range of transitions available in VideoDB Editor.

The three transition groups we'll show:
- **Fade** (classic opacity transition)
- **Reveal** (slide from edge)
- **Wipe** (sweeping effect from left and right)

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

track = Track()

# Clip 1: Boost filter + Fade transition
clip1 = Clip(
    asset=VideoAsset(id=video1.id, start=5),
    duration=7,
    filter=Filter.boost,
    transition=Transition(in_="fade", out="fade", duration=1.5)
)

# Clip 2: Boost filter + Reveal transition
clip2 = Clip(
    asset=VideoAsset(id=video2.id, start=10),
    duration=7,
    filter=Filter.boost,
    transition=Transition(in_="reveal", out="reveal", duration=1.5)
)

# Clip 3: Boost filter + Shuffle (center) transition
clip3 = Clip(
    asset=VideoAsset(id=video1.id, start=30),
    duration=7,
    filter=Filter.boost,
    transition=Transition(in_="wipe_left", out="wipe_left", duration=1.5)
)

track.add_clip(0, clip1)
track.add_clip(7, clip2)
track.add_clip(14, clip3)

timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/8a3e2be8-1b49-4c68-9b96-2e9b6cc3f99f.m3u8


> **What just happened?** This showcase combines a consistent filter (`boost`) across all clips while varying the transition type. Notice how each transition changes the feel:
> - **Fade** is smooth and classic
> - **Reveal** slides in dynamically
> - **Wipe** sweeps in from the side
> - **Shuffle** tiles in from the corner with a modern, energetic feel
>
> Mixing filter + transition combinations lets you create distinctive visual styles for different sections of your video.


---

## 📦 Step 16: Creating a Filter Showcase with Different Transitions

Let's create a more complex example: multiple clips with different filters, each with its own transition timing. This demonstrates how filters and transitions work together to create varied visual experiences.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

track = Track()

# Greyscale with long fade
clip1 = Clip(
    asset=VideoAsset(id=video1.id, start=5),
    duration=7,
    filter=Filter.greyscale,
    transition=Transition(in_="fade", out="fade", duration=2)
)

# Boost with quick fade
clip2 = Clip(
    asset=VideoAsset(id=video2.id, start=10),
    duration=7,
    filter=Filter.boost,
    transition=Transition(in_="fade", out="fade", duration=1)
)

# Negative with long fade
clip3 = Clip(
    asset=VideoAsset(id=video1.id, start=30),
    duration=7,
    filter=Filter.negative,
    transition=Transition(in_="fade", out="fade", duration=2)
)

track.add_clip(0, clip1)
track.add_clip(7, clip2)
track.add_clip(14, clip3)

timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Stream URL: https://play.videodb.io/v1/36f85c66-9756-4484-adcb-ecd3646bdafd.m3u8


<div style="background-color: #d4edda; color: #155724; padding: 12px; border-left: 5px solid #28a745; border-radius: 4px;">
    <strong>🎉 Success!</strong> We've demonstrated filters, transitions, and their combinations — from simple single-clip effects to complex multi-scene sequences with different transition timings.
</div>

> **What just happened?** Each clip in this showcase has a unique filter (greyscale, boost, negative) paired with different transition durations (2s, 1s, 2s). This is the real power of the Editor API — fine-grained control over every visual aspect of your composition.


**Notice:** Each clip has:
- A different **filter** (greyscale, boost, negative)
- A different **transition duration** (2s, 1s, 2s)
- A different **source video segment**

This creates a dynamic, visually interesting sequence where each scene has its own character.

<div style="background-color: #d4edda; color: #155724; padding: 12px; border-left: 5px solid #28a745; border-radius: 4px;">
    <strong>🏆 Congratulations!</strong> You've completed the Filters & Transitions notebook. You now know how to apply all 8 filters, use all 8 transition types (fade, reveal, wipe, and shuffle variants), combine them with other clip properties, and build multi-scene sequences with smooth transitions.
</div>

---

## 🎬 Wrap-Up: What We've Learned

Congratulations! We've covered the complete toolkit for filters and transitions in VideoDB Editor.

### Filters

We explored all 8 available filters and their effects:
- **Filter.greyscale** - Classic black-and-white look
- **Filter.blur** - Artistic softening or privacy effect
- **Filter.contrast** - Dramatic darks and lights
- **Filter.darken** - Reduce overall brightness
- **Filter.lighten** - Increase overall brightness
- **Filter.boost** - Vibrant colors with enhanced contrast
- **Filter.muted** - Subtle, understated colors
- **Filter.negative** - Surreal color inversion

### Transitions

We explored all **8 transition types**:

| Transition | Effect |
|-----------|--------|
| `fade` | Fade to/from black (classic smooth transition) |
| `reveal` | Slide reveal from edge |
| `wipe_left` | Wipe/sweep from the left edge |
| `wipe_right` | Wipe/sweep from the right edge |
| `shuffle_top_right` | Shuffle from top-right corner |
| `shuffle_bottom_right` | Shuffle from bottom-right corner |
| `shuffle_bottom_left` | Shuffle from bottom-left corner |
| `shuffle_top_left` | Shuffle from top-left corner |

**Key transition patterns:**
- **Fade in** (`in_="fade"`) - Smooth appearance
- **Fade out** (`out="fade"`) - Smooth disappearance
- **Duration control** - Customize transition length (recommended: 2 seconds)
- **Independent application** - Use fade-in, fade-out, or both

### Key Insights

1. **Clip-level effects**: Both filters and transitions are applied to Clips, not Assets. This means the same video can have different effects in different contexts.

2. **Combinations work seamlessly**: Filters, transitions, scale, opacity, and position can all be combined on a single clip for sophisticated effects.

3. **Professional polish without GUI**: We can create cinematic, polished videos entirely through code—no timeline scrubbing or effect panels needed.

4. **Works with all assets**: Filters and transitions apply to VideoAssets, ImageAssets, and even TextAssets.

### Practical Use Cases

- **Social media content**: Add greyscale or boost filters for consistent brand aesthetics
- **Video montages**: Use transitions for smooth scene changes
- **Artistic sequences**: Combine filters, transitions, and opacity for creative effects
- **Flashback scenes**: Greyscale + fade transitions for temporal storytelling
- **Slideshow presentations**: Image clips with fade transitions
- **Modern intros**: Reveal or shuffle transitions for dynamic opening sequences
- **Social media content**: Shuffle variants for energetic, attention-grabbing clips
- **Corner branding**: Use shuffle transitions from specific corners for branded content transitions

### What's Next?

Now that we understand filters and transitions, we can:
- Experiment with different filter combinations on the same video
- Create custom transition timing patterns for rhythmic editing
- Combine these effects with layered tracks for complex compositions
- Build reusable templates for common video styles

Happy editing! 🎥✨